In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("hotel_bookings.csv")

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27.0,1.0,0.0,0.0,2.0,...,No Deposit,NaN,NaN,0.0,Transient,0.0,0.0,0.0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27.0,1.0,0.0,0.0,2.0,...,No Deposit,NaN,NaN,0.0,Transient,0.0,0.0,0.0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27.0,1.0,0.0,1.0,1.0,...,No Deposit,NaN,NaN,0.0,Transient,75.0,0.0,0.0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27.0,1.0,0.0,1.0,1.0,...,No Deposit,304.0,NaN,0.0,Transient,75.0,0.0,0.0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27.0,1.0,0.0,2.0,2.0,...,No Deposit,240.0,NaN,0.0,Transient,98.0,0.0,1.0,Check-Out,2015-07-03


In [2]:
print(df.shape)

print(df.info())

print(df.isnull().sum())

(7324, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7324 entries, 0 to 7323
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           7324 non-null   object 
 1   is_canceled                     7324 non-null   int64  
 2   lead_time                       7324 non-null   int64  
 3   arrival_date_year               7324 non-null   int64  
 4   arrival_date_month              7324 non-null   object 
 5   arrival_date_week_number        7323 non-null   float64
 6   arrival_date_day_of_month       7323 non-null   float64
 7   stays_in_weekend_nights         7323 non-null   float64
 8   stays_in_week_nights            7323 non-null   float64
 9   adults                          7323 non-null   float64
 10  children                        7323 non-null   float64
 11  babies                          7323 non-null   float64
 12  meal                   

In [3]:
df = df.drop(
    columns=[
        "reservation_status",
        "reservation_status_date"
    ]
)

In [4]:
df["children"] = df["children"].fillna(0)

df["country"] = df["country"].fillna("Unknown")

df["agent"] = df["agent"].fillna(0)

df["company"] = df["company"].fillna(0)

In [5]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:

    df[col] = encoder.fit_transform(df[col].astype(str))

In [6]:
X = df.drop("is_canceled", axis=1)

y = df["is_canceled"]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [9]:
from sklearn.metrics import accuracy_score

rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(
    y_test,
    rf_pred
)

print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.9419795221843004


In [14]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = df.fillna(0)
X = df.drop("is_canceled", axis=1)
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

gb = GradientBoostingClassifier(
    random_state=42
)

gb.fit(X_train, y_train)


GradientBoostingClassifier(random_state=42)

In [13]:
gb_pred = gb.predict(X_test)

gb_acc = accuracy_score(
    y_test,
    gb_pred
)

print("Gradient Boost Accuracy:", gb_acc)

Gradient Boost Accuracy: 0.9235494880546075


In [15]:
print("Random Forest :", rf_acc)

print("Gradient Boost :", gb_acc)

Random Forest : 0.9419795221843004
Gradient Boost : 0.9235494880546075


In [16]:
import matplotlib.pyplot as plt

importance = rf.feature_importances_

features = X.columns

importance_df = pd.DataFrame({

    "Feature": features,
    "Importance": importance

})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.head(10))

                        Feature  Importance
12                      country    0.280803
1                     lead_time    0.089376
2             arrival_date_year    0.071318
4      arrival_date_week_number    0.066522
26                          adr    0.066040
27  required_car_parking_spaces    0.058061
22                        agent    0.037012
5     arrival_date_day_of_month    0.034660
13               market_segment    0.032317
3            arrival_date_month    0.030358


In [17]:
import pickle

pickle.dump(
    rf,
    open("model.pkl", "wb")
)

pickle.dump(
    list(X.columns),
    open("columns.pkl", "wb")
)

In [18]:
import pickle

pickle.dump(
    rf,
    open("model.pkl", "wb")
)

pickle.dump(
    list(X.columns),
    open("columns.pkl", "wb")
)

In [19]:
!ls

columns.pkl  hotel_bookings.csv  model.pkl  sample_data
